In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
import numpy as np
from numpy.typing import NDArray
from src.allocation import pcaweights
from src.etf_trick import ETFTrick
from src.data_processing import multi_assets_synchro_multi
from src.instrument_config import FUTURE_SPECS
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
rolling_dates, etf_data = multi_assets_synchro_multi(["ES", "NQ", "YM", "CL", "RB"])

In [ ]:
etf_data.head()

In [ ]:
close_col = [col for col in etf_data.columns if col.startswith('close_')]
etf_data_close = etf_data[close_col]

In [ ]:
def get_robust_covariance(high_freq_df: pd.DataFrame, freq='5min') -> NDArray[np.float64]:
    """
    Calcule une covariance robuste en rééchantillonnant et en gérant proprement les données manquantes.
    """
    # 1. Sélection des colonnes Close
    close_cols = [c for c in high_freq_df.columns if c.startswith('close_')]
    df_close = high_freq_df[close_cols]
    
    # 2. Resampling sur une grille fixe
    # Attention: .last() peut générer des NaN si aucun trade n'a eu lieu pendant 5 min
    df_resampled = df_close.resample(freq).last()
    
    # 3. Remplissage Explicite (La correction du Warning)
    # Si un actif ne cot pas pendant 5 min, son prix reste le même que la veille (Forward Fill).
    # C'est financièrement logique : rendement = 0%.
    df_resampled = df_resampled.ffill()
    print(df_resampled)
    
    # 4. Calcul des rendements
    # On ajoute fill_method=None pour dire à Pandas : "T'inquiète, j'ai déjà géré les NaN avant"
    returns = df_resampled.pct_change(fill_method=None).dropna()
    
    # 5. Covariance NumPy
    cov_matrix = np.cov(returns.values, rowvar=False)
    
    return cov_matrix

In [ ]:
cov_matrix = get_robust_covariance(etf_data)

In [ ]:
weights = pcaweights(cov_matrix, np.array([0, 1/4, 1/4, 1/4, 1/4]))

In [ ]:
cov_matrix

In [ ]:
weights

In [ ]:
etf_trick = ETFTrick(FUTURE_SPECS)

In [ ]:
etf_trick = etf_trick.run(etf_data, rolling_dates, weights)

In [ ]:
plt.figure(figsize=(15, 10))
etf_trick.K.plot()

In [ ]:
print(cov_matrix)